# Example 4B: reviewed-analysis stability checks

This notebook follows directly from Example 4A. It keeps the **same X-SHOOTER UVB spectrum** and asks whether the baseline Balmer-wing interpretation remains stable when we change reasonable analysis choices.

## What this example teaches

- how to test whether a baseline Balmer result is stable under nearby analysis choices;
- how to compare parameter shifts from continuum, core-mask, line-selection, and resolution variants;
- how to annotate suspicious residual regions without automatically correcting or masking them.

## Requirements

The planning cells use bundled data only. Running the baseline and stability variants requires PHOENIX and can take noticeably longer than Examples 1–3.

## Expected outputs

A planned-variant table, optional baseline and variant fits, compact comparison tables, and residual/feature-triage plots.

Note: A stability suite measures sensitivity to the tested choices only; it is not a complete external systematic-error calibration.


Example 4A prepared and audited a baseline Balmer case. Here we test a small set of follow-up choices recommended at the end of 4A:

1. change the residual continuum degree;
2. change the Balmer-core mask width;
3. remove one Balmer line at a time;
4. perturb the constant Gaussian resolution/LSF assumption.

These are sensitivity checks. If a result moves substantially between reasonable variants, that tells us what needs deeper review before scientific interpretation. A stable table is encouraging, but it is still not a complete uncertainty budget.


## 0. Controls

The controls below decide how far the notebook runs. Set `RUN_BASELINE_FIT = False` and `RUN_STABILITY_VARIANTS = False` for a review-only pass that does not load PHOENIX. Turn them on only after the preparation plots make sense.

`MAX_VARIANTS = None` means “run every planned variant.” Use an integer, for example `MAX_VARIANTS = 3`, only when you want a short editing/smoke-test pass.


In [ ]:
import Spyctres as sp

# Use None to let Spyctres find PHOENIX from its normal configuration.
PHOENIX_DIR = None

# Print elapsed-time progress during the PHOENIX cache, RV scan, and optimizers.
SHOW_PROGRESS = True

# Keep these False for a quick read-through. Set them deliberately when ready.
RUN_BASELINE_FIT = True
RUN_STABILITY_VARIANTS = True

# None runs all planned variants. Use an integer for a short subset while editing.
MAX_VARIANTS = None

# These are analysis choices for this notebook, not global Spyctres defaults.
# They mirror the reviewed Balmer setup from Example 4A.
baseline_core_mask_A = 8.0
baseline_continuum_degree = 1


## 1. Reload the Example 4A spectrum

We re-use the same bundled X-SHOOTER UVB product and the same reader.


In [ ]:
spectrum_path = sp.example_data_path("TOO_Gaia21ccu_SCI_SLIT_FLUX_MERGE1D_UVB.fits")
reader = "xshooter_merge1d"

spec = sp.read_spectrum(spectrum_path, reader=reader)
print(spec.summary())
print(spec.provenance_summary())


## 2. Rebuild the baseline Balmer case

Let's recreate the same preparation from Example 4A: sideband normalization, Hδ/Hγ/Hβ windows, and a Balmer-core mask. This gives every later variant a clear baseline.

The recipe is used here because Example 4A already explained the construction step-by-step. In 4B we are testing stability of reviewed choices, not re-teaching sideband normalization from scratch.


In [ ]:
baseline_case = sp.recipes.prepare_xshooter_balmer_case(
    spec,
    window_mode="notebook",
    norm_mode="sideband",
    sideband_width=10.0,
    sideband_order=1,
    core_mask=baseline_core_mask_A,
)

print(baseline_case.summary_text())

baseline_case.plot_preparation(
    title="Example 4B: baseline Balmer preparation from Example 4A",
    ncols=2,
    figsize_per_panel=(7.2, 3.6),
)


## 3. Build the same reviewed-analysis intent setup and audit

The audit tells us whether the setup is interpretable as analysis-ready. If it is blocked, we may still run exploratory fits for diagnosis, but the result should stay labelled as exploratory.


In [ ]:
# Build a FitSetup for the baseline Balmer analysis.
#
# mode controls the search budget:
#   quicklook   = fast first pass
#   standard    = moderate reviewed first pass
#   diagnostic  = wider stress/debug search
#
# intent controls the readiness gate:
#   reviewed_analysis asks whether the metadata, masks, resolution,
#   uncertainties, and fit window are explicit enough to interpret the result.
#
# continuum_degree controls the multiplicative Legendre continuum fitted
# inside the PHOENIX model comparison. Degree 1 is a linear correction;
# degree 2 allows curvature but can absorb more astrophysical signal.
baseline_setup = baseline_case.suggest_fit_setup(
    mode="standard",
    intent="reviewed_analysis",
    continuum_degree=baseline_continuum_degree,
)

# Run the stricter reviewed-analysis audit on the same prepared Balmer windows.
# This does not fit anything. It only checks whether the selected pixels,
# metadata, uncertainty information, resolution information, and masks are
# good enough for cautious interpretation.
baseline_audit = sp.analysis_readiness_audit(
    baseline_case.collection,
    regions=baseline_case.fit_regions_by_segment,
    exclude_masks=baseline_case.exclusion_masks,
)

print(baseline_setup.summary_text(include_hash=False))
print("Reviewed-analysis ready:", baseline_audit["analysis_ready"])
print("Reviewed-analysis blockers:", baseline_audit["blockers"])
print("Reviewed-analysis warnings:", baseline_audit["warnings"])


If you want to see the available options programmatically, use `sp.help("suggest_fit_setup")` and `sp.help("analysis_readiness_audit")`.

## 4. Define bounded variants

Each variant changes one reviewed choice while leaving the rest of the workflow alone. That is important because if we change window selection, continuum handling, masks, and resolution all at once, we cannot tell which choice caused a parameter shift.

The variant list is explicit so a user can see exactly which scientific question each fit is asking before spending time running PHOENIX.


In [ ]:
# Prepare two alternative Balmer cases with narrower and wider core masks.
# The baseline mask is varied to test how strongly the fitted parameters
# depend on the amount of central line profile that is excluded.

core4_case = sp.recipes.prepare_xshooter_balmer_case(
    spec,
    window_mode="notebook",
    norm_mode="sideband",
    sideband_width=10.0,  # Fallback value; notebook mode uses predefined sidebands.
    sideband_order=1,     # Fit a straight local continuum through the sidebands.
    core_mask=4.0,        # Exclude ±4 Å around each Balmer-line centre.
)

core10_case = sp.recipes.prepare_xshooter_balmer_case(
    spec,
    window_mode="notebook",
    norm_mode="sideband",
    sideband_width=10.0,
    sideband_order=1,
    core_mask=10.0,       # Exclude a wider ±10 Å central region.
)


# Find the segment positions needed to construct two-line subsets.
# These tests show whether either Hδ or Hβ has excessive influence
# on the joint three-line solution.
hgamma_hbeta_indices = [
    i
    for i, segment in enumerate(baseline_case.fit_segments)
    if segment.name in {"Hγ", "Hβ"}
]

hdelta_hgamma_indices = [
    i
    for i, segment in enumerate(baseline_case.fit_segments)
    if segment.name in {"Hδ", "Hγ"}
]


# Build collections containing only the selected Balmer lines.
hgamma_hbeta_collection = sp.SpectrumCollection(
    [baseline_case.fit_segments[i] for i in hgamma_hbeta_indices],
    name="example4b_without_hdelta",
)

hdelta_hgamma_collection = sp.SpectrumCollection(
    [baseline_case.fit_segments[i] for i in hdelta_hgamma_indices],
    name="example4b_without_hbeta",
)


# Select the matching masks, plotting windows, and fitting regions.
# Their order must remain aligned with the segments in each collection.
hgamma_hbeta_masks = tuple(
    baseline_case.valid_masks[i] for i in hgamma_hbeta_indices
)
hdelta_hgamma_masks = tuple(
    baseline_case.valid_masks[i] for i in hdelta_hgamma_indices
)

hgamma_hbeta_windows = tuple(
    baseline_case.fit_windows[i] for i in hgamma_hbeta_indices
)
hdelta_hgamma_windows = tuple(
    baseline_case.fit_windows[i] for i in hdelta_hgamma_indices
)

hgamma_hbeta_regions = tuple(
    baseline_case.fit_regions[i] for i in hgamma_hbeta_indices
)
hdelta_hgamma_regions = tuple(
    baseline_case.fit_regions[i] for i in hdelta_hgamma_indices
)


In [ ]:
# Each variant changes one analysis choice while keeping the others
# as close as possible to the baseline fit. Let's define so variants to check.
variant_plan = [
    {
        "id": "continuum_degree_0",
        "label": "continuum degree 0",
        "question": (
            "Does the solution change when no residual continuum slope "
            "is allowed?"
        ),
        "collection": baseline_case.collection,
        "valid_mask": baseline_case.valid_masks,
        "windows": baseline_case.fit_windows,
        "setup": baseline_case.suggest_fit_setup(
            mode="standard",
            intent="reviewed_analysis",
            continuum_degree=0,
        ),
    },
    {
        "id": "continuum_degree_2",
        "label": "continuum degree 2",
        "question": (
            "Does a more flexible continuum change the Balmer-wing solution?"
        ),
        "collection": baseline_case.collection,
        "valid_mask": baseline_case.valid_masks,
        "windows": baseline_case.fit_windows,
        "setup": baseline_case.suggest_fit_setup(
            mode="standard",
            intent="reviewed_analysis",
            continuum_degree=2,
        ),
    },
    {
        "id": "core_mask_4A",
        "label": "Balmer core mask ±4 Å",
        "question": (
            "Do the parameters shift when more of the inner line profile "
            "is included?"
        ),
        "collection": core4_case.collection,
        "valid_mask": core4_case.valid_masks,
        "windows": core4_case.fit_windows,
        "setup": core4_case.suggest_fit_setup(
            mode="standard",
            intent="reviewed_analysis",
            continuum_degree=baseline_continuum_degree,
        ),
    },
    {
        "id": "core_mask_10A",
        "label": "Balmer core mask ±10 Å",
        "question": (
            "Do the parameters shift when more of the core and inner wings "
            "are excluded?"
        ),
        "collection": core10_case.collection,
        "valid_mask": core10_case.valid_masks,
        "windows": core10_case.fit_windows,
        "setup": core10_case.suggest_fit_setup(
            mode="standard",
            intent="reviewed_analysis",
            continuum_degree=baseline_continuum_degree,
        ),
    },
    {
        "id": "without_hdelta",
        "label": "without Hδ",
        "question": (
            "Is the solution stable after removing the line with the largest "
            "reader-rejected interval?"
        ),
        "collection": hgamma_hbeta_collection,
        "valid_mask": hgamma_hbeta_masks,
        "windows": hgamma_hbeta_windows,
        "setup": baseline_case.suggest_fit_setup(
            mode="standard",
            intent="reviewed_analysis",
            continuum_degree=baseline_continuum_degree,
        ).with_regions(hgamma_hbeta_regions),
    },
    {
        "id": "without_hbeta",
        "label": "without Hβ",
        "question": (
            "Is the solution stable after removing the strongest available "
            "Balmer line?"
        ),
        "collection": hdelta_hgamma_collection,
        "valid_mask": hdelta_hgamma_masks,
        "windows": hdelta_hgamma_windows,
        "setup": baseline_case.suggest_fit_setup(
            mode="standard",
            intent="reviewed_analysis",
            continuum_degree=baseline_continuum_degree,
        ).with_regions(hdelta_hgamma_regions),
    },
    {
        "id": "resolution_R5000",
        "label": "constant R=5000",
        "question": (
            "Does adopting a somewhat lower resolving power change the result?"
        ),
        "collection": baseline_case.collection,
        "valid_mask": baseline_case.valid_masks,
        "windows": baseline_case.fit_windows,
        "setup": baseline_case.suggest_fit_setup(
            mode="standard",
            intent="reviewed_analysis",
            continuum_degree=baseline_continuum_degree,
        ).with_resolution(R=5000.0),
    },
    {
        "id": "resolution_R6200",
        "label": "constant R=6200",
        "question": (
            "Does adopting a somewhat higher resolving power change the result?"
        ),
        "collection": baseline_case.collection,
        "valid_mask": baseline_case.valid_masks,
        "windows": baseline_case.fit_windows,
        "setup": baseline_case.suggest_fit_setup(
            mode="standard",
            intent="reviewed_analysis",
            continuum_degree=baseline_continuum_degree,
        ).with_resolution(R=6200.0),
    },
]


# Select the variants that will actually run. None means all variants.
if MAX_VARIANTS is None:
    selected_variant_plan = list(variant_plan)
else:
    selected_variant_plan = list(variant_plan[:int(MAX_VARIANTS)])

omitted_variant_plan = variant_plan[len(selected_variant_plan):]

print("Planned sensitivity tests:")
for item in variant_plan:
    print(f"- {item['label']}: {item['question']}")

print(
    f"\nWhen RUN_STABILITY_VARIANTS=True, this notebook will run "
    f"{len(selected_variant_plan)}/{len(variant_plan)} variants."
)
if omitted_variant_plan:
    print("Omitted by MAX_VARIANTS:")
    for item in omitted_variant_plan:
        print(f"- {item['label']}")


## 5. Optional baseline fit

If the audit is blocked, the code below records an explicit exploratory override. Practically, this means Spyctres will compute the fit so we can inspect residuals, but the result summary and provenance will say that the fit is **not final analysis**.

Here the baseline is the reference against which every later variant is compared. We do not choose the “best” variant by eye; we use the variant suite to see which assumptions the result is sensitive to.


In [ ]:
baseline_result = None
setup_for_fit = baseline_setup

if RUN_BASELINE_FIT or RUN_STABILITY_VARIANTS:
    # If the audit were blocked, this would keep the computation honest:
    # Spyctres may run the fit, but labels the interpretation as exploratory.
    if baseline_audit["analysis_ready"] is not True:
        setup_for_fit = baseline_setup.allow_exploratory(
            reason="Example 4B follow-up stability check; not final analysis."
        )

    baseline_result = sp.fit_stellar_spectrum(
        baseline_case.collection,
        model="phoenix",
        setup=setup_for_fit,
        valid_mask=baseline_case.valid_masks,
        phoenix_dir=PHOENIX_DIR,
        progress_callback=(
            (lambda event: print(f"[{event.elapsed_s:6.1f}s] {event}", flush=True))
            if SHOW_PROGRESS
            else None
        ),
    )
    print(baseline_result.summary_text(include_hash=False, max_flags=8))

    # Inspect the baseline fit line by line before trusting the number table.
    sp.plot_model_line_windows(
        baseline_result,
        windows=baseline_case.fit_windows,
        title="Example 4B: baseline Balmer fit",
        show_residuals=True,
        residual_kind="pull",
        ncols=2,
        figsize_per_panel=(7.2, 5.2),
    )
else:
    print("Set RUN_BASELINE_FIT=True to run the baseline PHOENIX fit.")


## 6. Optional stability suite

The goal is not to find the lowest χ² by trying every possible setup. The goal is to learn whether the inference changes under scientifically reasonable alternatives.

If `MAX_VARIANTS` is an integer, the comparison table will only include that subset. If you want the full table, set `MAX_VARIANTS = None` in the controls cell before running this section.


In [ ]:
variant_records = []
variant_results = []
variant_labels = []
variant_parameter_rows = []
stability_comparison = None
comparison_results = []
comparison_labels = []

if RUN_STABILITY_VARIANTS:
    if baseline_result is None:
        raise ValueError("Run or create the baseline result before comparing variants.")

    print(
        f"Running {len(selected_variant_plan)}/{len(variant_plan)} planned variants."
    )

    for index, item in enumerate(selected_variant_plan, start=1):
        variant_setup = item["setup"]
        if baseline_audit["analysis_ready"] is not True:
            variant_setup = variant_setup.allow_exploratory(
                reason="Example 4B follow-up stability variant; not final analysis."
            )

        print("\n" + item["label"])
        print(item["question"])

        # This is the expensive operation in the stability section.
        # The resulting PhoenixFitResult is stored so later cells can format
        # tables and plots without rerunning PHOENIX.
        result = sp.fit_stellar_spectrum(
            item["collection"],
            model="phoenix",
            setup=variant_setup,
            valid_mask=item["valid_mask"],
            phoenix_dir=PHOENIX_DIR,
            progress_callback=(
                (lambda event: print(f"[{event.elapsed_s:6.1f}s] {event}", flush=True))
                if SHOW_PROGRESS
                else None
            ),
        )

        variant_records.append(
            {
                "id": item["id"],
                "label": item["label"],
                "question": item["question"],
                "result": result,
                "windows": item["windows"],
            }
        )
        variant_parameter_rows.append(
            {
                "id": item["id"],
                "label": item["label"],
                "teff": result.summary.get("teff"),
                "logg": result.summary.get("logg"),
                "feh": result.summary.get("feh"),
                "rv_kms": result.summary.get("rv_kms"),
                "chi2_red": result.summary.get("chi2_red"),
                "success": result.summary.get("success"),
            }
        )
        variant_results.append(result)
        variant_labels.append(item["label"])
        print(f"Stored variant {index}/{len(selected_variant_plan)}: {item['label']}")

    if omitted_variant_plan:
        print("\nNot run in this pass because MAX_VARIANTS limited the suite:")
        for item in omitted_variant_plan:
            print(f"- {item['label']}: {item['question']}")
else:
    print("Set RUN_STABILITY_VARIANTS=True after inspecting the baseline fit.")


## 7. Summarize the stored stability results

This cell is intentionally cheap: it only formats the results already stored by the previous cell. You can rerun it while editing the notebook without launching another PHOENIX fit. The plain parameter dictionaries are also available in `variant_parameter_rows` if you want to inspect or export them manually.

The table compares the baseline result with each variant. Look for parameter shifts that are large compared with the scientific precision you need. A low χ² alone is not enough; a result also needs stable parameters and acceptable residuals.

For a quick notebook sanity check, shifts of order several hundred kelvin in Teff, several tenths of a dex in logg/[Fe/H], or several km/s in RV are worth investigating. Those are not universal acceptance thresholds; your science case and external validation set define the final tolerance.


In [ ]:
stability_comparison = None
comparison_results = []
comparison_labels = []

if baseline_result is not None and variant_results:
    comparison_results = [baseline_result] + variant_results
    comparison_labels = ["baseline"] + variant_labels

    stability_comparison = sp.compare_fits(
        comparison_results,
        labels=comparison_labels,
    )

    print(sp.format_fit_comparison_table(stability_comparison))
    print(
        "\nHow to read this table: each row changes one analysis choice "
        "relative to the baseline. Stable parameters across rows are "
        "encouraging; large shifts identify the assumption that needs review. "
        "The status and grid/bounds columns matter as much as chi-square."
    )

    if omitted_variant_plan:
        print("\nVariants not included in this table because MAX_VARIANTS limited the suite:")
        for item in omitted_variant_plan:
            print(f"- {item['label']}: {item['question']}")
elif RUN_STABILITY_VARIANTS:
    print("Run the previous cell first to create variant_results.")
else:
    print("Set RUN_STABILITY_VARIANTS=True and run the previous cell first.")


## 8. Plot baseline and variants together

A table shows parameter shifts, but an overplot shows *where* the fits differ. This is especially useful for Balmer wings: two variants can have similar parameter values but visibly disagree in one line window.

This cell uses the stored `comparison_results` from the table cell above. It should not rerun any fits.

If the plot becomes too busy, run a smaller subset by setting `MAX_VARIANTS` to an integer and repeat the run/table/plot cells.


In [ ]:
if comparison_results:
    sp.plot_fit_comparison_line_windows(
        comparison_results,
        labels=comparison_labels,
        windows=baseline_case.fit_windows,
        title="Example 4B: baseline and stability variants overplotted",
        ncols=2,
        figsize_per_panel=(7.2, 3.8),
        footer=(
            "Large separations between traces show sensitivity to "
            "reasonable analysis choices."
        ),
    )
else:
    print("Run the stability-variant and table cells first.")


## 9. Triage unexplained residual structures

Spyctres also offers some tools to help you **diagnose** features in the residuals.

The workflow below is deliberately cautious:

1. write down the approximate wavelength intervals where the residuals look suspicious;
2. ask Spyctres whether any known broad non-stellar catalog features overlap those intervals;
3. mark candidate catalog regions on the observed/model residual plot;
4. quantify whether the residuals in those curated windows are coherent enough to flag;
5. decide what follow-up sensitivity test, if any, is justified.

A catalog overlap is only a clue, but it does not prove that the residual is definitely a DIB, a telluric band, or any other specific feature.


In [ ]:
candidate_feature_matches = []
known_feature_overlap = None
known_residual_windows = None

# Let's here define some residual intervals that we noticed. Approximate ranges can be given.
# In a new spectrum, you would change these intervals to whatever looks suspicious.
suspicious_residual_regions = [
    {
        "label": "unexplained dip near Hγ red wing",
        "region_A": (4415.0, 4445.0),
    },
    {
        "label": "unexplained Hβ red-wing drop",
        "region_A": (4875.0, 4910.0),
    },
]

if baseline_result is not None:
    # First ask: does Spyctres know any broad non-stellar catalog features
    # that overlap the wavelength intervals the user noticed?
    candidate_feature_matches = sp.find_known_nonstellar_features(
        suspicious_residual_regions,
        padding_A=0.0,
    )

    print("Candidate catalog overlaps for the user-noticed residual regions:")
    if candidate_feature_matches:
        for match in candidate_feature_matches:
            label = match.get("query_label") or f"region {match['query_index']}"
            print(
                f"- {label}: {match['name']} "
                f"({match['id']}), catalog region={match['region_A']} Å, "
                f"overlap={match['overlap_A']:.1f} Å"
            )
            print(f"  note: {match['note']}")
    else:
        print("- no known broad non-stellar catalog feature overlaps these intervals")

    # If there are candidate matches, record them in the fit result provenance.
    # policy="warn" means: flag/annotate only. No pixels are masked here.
    candidate_feature_ids = tuple(
        dict.fromkeys(match["id"] for match in candidate_feature_matches)
    )
    if candidate_feature_ids:
        known_feature_overlap = sp.annotate_nonstellar_features(
            baseline_case.collection,
            baseline_result,
            feature_names=candidate_feature_ids,
            policy="warn",
            verbose=True,
        )

    # Then ask whether curated residual windows linked to known broad features
    # are coherent enough to be worth flagging. This still does not identify
    # the physical cause; it only highlights the need for manual follow-up inspection.
    known_residual_windows = sp.diagnose_known_residual_windows(
        baseline_case.collection,
        baseline_result,
        threshold_sigma=2.5,
        verbose=True,
    )

    print("\nCurated residual-window flags:")
    if known_residual_windows["flagged_windows"]:
        for window in known_residual_windows["flagged_windows"]:
            print(
                f"- {window['name']}: median={window['median_sigma']:.2f}σ, "
                f"rms={window['rms_sigma']:.2f}σ; "
                f"hypothesis={window['origin_hypothesis']}"
            )
            print(f"  action: {window['recommended_action']}")
    else:
        print("- none above the chosen threshold")

    # Plot the full prepared Balmer line windows, not just the suspicious
    # residual intervals. This keeps the line-wing context visible while the
    # candidate catalog overlaps are highlighted in orange.
    triage_line_windows = []
    for window in baseline_case.fit_windows:
        wmin, wmax = window["limits_A"]
        overlaps_candidate = any(
            max(wmin, match["region_A"][0]) < min(wmax, match["region_A"][1])
            for match in candidate_feature_matches
        )
        if overlaps_candidate:
            triage_line_windows.append(window)

    # If no catalog match was found, still show the line windows that contain
    # the user-noticed residual intervals. The plot remains diagnostic-only.
    if not triage_line_windows:
        for window in baseline_case.fit_windows:
            wmin, wmax = window["limits_A"]
            overlaps_user_region = any(
                max(wmin, region["region_A"][0]) < min(wmax, region["region_A"][1])
                for region in suspicious_residual_regions
            )
            if overlaps_user_region:
                triage_line_windows.append(window)

    sp.plot_model_line_windows(
        baseline_result,
        windows=triage_line_windows,
        annotation_regions=candidate_feature_matches,
        title="Example 4B: residual triage with candidate catalog features",
        show_residuals=True,
        residual_kind="pull",
        ncols=2,
        figsize_per_panel=(7.2, 5.0),
        footer=(
            "Orange = candidate catalog feature overlap, not a mask or correction. "
            "Inspect before deciding whether to run a controlled sensitivity test."
        ),
    )
else:
    print("Run the baseline fit first, then rerun this diagnostic cell.")


### How to interpret the residual-triage plot

The orange bands are **candidate catalog overlaps**, not pixels that Spyctres removed. They answer the question: “is there something known in this approximate wavelength range that I should consider?”

If a candidate region is flagged, the next reasonable follow-up options are:

- inspect the residuals in other Balmer lines; is there a common pattern?
- check whether the feature is fixed in the observed/topocentric frame or follows the stellar rest frame;
- test continuum degree, core-mask width, and resolution/LSF sensitivity;
- optionally run a controlled named-mask variant with `sp.known_feature_masks(...)` and compare with `sp.compare_fits()`.

Only treat a feature as an identified contaminant if your analysis supports that interpretation. Otherwise, report it as a systematic sensitivity or quality flag.


## 10. How to read the result

A stable result means the fitted Teff/logg/[Fe/H]/RV do not move much when you adjust continuum degree, core-mask width, line selection, or plausible constant-R assumptions.

An unstable result is useful too: it tells you whether the limiting issue is continuum placement, the Balmer-core treatment, one problematic line, resolution/LSF assumptions, or candidate non-stellar residual features.

What this notebook can support:

- a reviewer-friendly statement that the baseline fit was tested against a bounded set of reasonable alternatives;
- a list of assumptions that materially affect the answer;
- diagnostic feature annotations that tell the user what to inspect next.

What this notebook cannot support by itself:

- a full systematic uncertainty budget;
- empirical wavelength-dependent LSF calibration;
- external calibration against benchmark stars or overlapping instruments.

The next step after this notebook is Example 6: multiband classification, where the parameters are checked for UVB/VIS/NIR consistency.
